# Laplace operator in JAX

In [1]:
import jax
import jax.numpy as jnp
from jax import grad, vmap, jit

def laplace(func, x):
    """Compute the Laplace operator of the model output with respect to inputs."""
    grad_fn = grad(func)
    d2_dx2 = 0
    for i in range(x.shape[1]):
        d2_dx2 += vmap(grad(lambda xi: grad_fn(xi)[i]))(x)[:, i]
    return d2_dx2

def model(x):
    # x is (batch_size, input_dim)
    return jnp.sum(x**3, axis=-1) 

def solution(x):
    return jnp.sum(6 * x, axis=-1)

# Example usage

x = jax.random.uniform(jax.random.PRNGKey(0), (10, 2))  # 1000 samples, 3 dimensions
# x = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0], [9.0, 10.0]])
print(f"Input size: {x.shape}")
laplace_values = laplace(model, x)
solution_values = solution(x)
print(laplace_values)
print(solution_values)

Input size: (10, 2)
[11.557481   4.8057594  4.4123507  5.9980507  5.5066867  6.0638185
  7.2163963 11.002508   9.358538   3.2250316]
[11.557481   4.8057594  4.4123507  5.9980507  5.5066867  6.0638185
  7.2163963 11.002508   9.358538   3.2250316]


In [10]:
import flax.linen as nn
# Define a simple MLP using Flax Linen
class MLP(nn.Module):
    architecture: list
    hidden_activation: callable = nn.tanh

    @nn.compact
    def __call__(self, x):
        for i in range(len(self.architecture) - 1):
            x = nn.Dense(features=self.architecture[i + 1])(x)
            if i < len(self.architecture) - 2:
                x = self.hidden_activation(x)

        return x

In [40]:
def laplace(func, x):
    """Compute the Laplace operator of the model output with respect to inputs."""
    grad_fn = jax.grad(func)
    d2_dx2 = 0
    for i in range(x.shape[1]):
        d2_dx2 += jax.vmap(jax.grad(lambda xi: grad_fn(xi)[i]))(x)[:, i]
    return d2_dx2

In [49]:
def local_energy_batch(params, xs, model_apply):
    # xs: (batch, 1) or (batch,)

    # psi(x) -> scalar
    def psi_fn(x):
        # ensure input has shape (1,) as model expects last-dim features
        return model_apply(params, x).squeeze()

    # second derivative per sample via AD
    psi_vals = jax.vmap(psi_fn)(xs)  # shape (batch,)
    print("psi_vals shape:", psi_vals.shape)
    d2psi = laplace(psi_fn, xs)  # shape (batch,)
    print("d2psi shape:", d2psi.shape)
    # avoid division by zero / small psi
    psi_safe = psi_vals + 1e-12

    kinetic = -0.5 * (d2psi / psi_safe)  # shape (batch,)
    print("kinetic shape:", kinetic.shape)
    potential = jnp.sum(0.5 * (xs**2), axis=-1)  # shape (batch,)
    print("potential shape:", potential.shape)
    return (kinetic + potential).reshape(-1, 1)  # keep your (batch,1) convention

In [51]:
features = [5, 3, 1]
model_instance = MLP(architecture=features)
batch_size = 1000
shape = (batch_size, features[0])
params = model_instance.init(jax.random.PRNGKey(0), jnp.ones(shape))  # initialize with dummy input
def print_params_shapes(params, prefix='0'):
    if isinstance(params, dict):
        for k, v in params.items():
            print_params_shapes(v, prefix + '/' + k)
    else:
        print(f"{prefix}: {params.shape}")

print_params_shapes(params)
model_apply = model_instance.apply
batch = jax.random.uniform(jax.random.PRNGKey(1), shape, minval=-5.0, maxval=5.0)
print("Batch shape:", batch.shape)
test_forward_pass = model_apply(params, batch)
print("Test forward pass shape:", test_forward_pass.shape)
local_energy_array = local_energy_batch(params, batch, model_apply)
print("Local energy shape:", local_energy_array.shape)

0/params/Dense_0/kernel: (5, 3)
0/params/Dense_0/bias: (3,)
0/params/Dense_1/kernel: (3, 1)
0/params/Dense_1/bias: (1,)
Batch shape: (1000, 5)
Test forward pass shape: (1000, 1)
psi_vals shape: (1000,)
d2psi shape: (1000,)
kinetic shape: (1000,)
potential shape: (1000,)
Local energy shape: (1000, 1)


In [32]:
batch = jnp.array([[1.0, 2.0, 3.0],
                   [4.0, 5.0, 6.0],
                   [7.0, 8.0, 9.0],
                   [10.0, 11.0, 12.0]])  # shape (4, 3)
bias = jnp.array([1.0, 2.0, 3.0, 4.0]).reshape(-1, 1)  # shape (4, 1)

print(batch.shape)
print(bias.shape)

print(batch + bias)  # broadcasting works

(4, 3)
(4, 1)
[[ 2.  3.  4.]
 [ 6.  7.  8.]
 [10. 11. 12.]
 [14. 15. 16.]]
